# Matbench Steels: Small-Data Materials Property Prediction

**Project direction:** evaluate tabular ML models for small-data materials property prediction using Matbench.

This notebook follows the style of the reference lab notebooks in `ref_lab_notebook/`:

- Lab 1: load data, inspect variables, train baseline ML models, evaluate predictions.
- Lab 2: featurize materials compositions and test feature-reduction ideas.

Here we apply that workflow to `matbench_steels`, a small Matbench regression task for predicting steel yield strength from composition.

## Summary of key ideas

- Use the official Matbench folds instead of a random train/test split.
- Convert chemical formulas into numeric Magpie composition features with `matminer`.
- Train a Random Forest baseline and measure MAE/R2 on each fold.
- Add one simple extension inspired by Lab 2: leakage-free feature selection using Random Forest feature importance.
- Include an optional TabPFN section. It runs only if `TABPFN_TOKEN` is available, because TabPFN requires one-time license acceptance before downloading model weights.

## Local setup

This notebook is intended to run inside the project Conda environment:

```bash
cd "/Users/zuoming/Documents/New project/matbench-tabpfn"
conda activate matbench-tabpfn
jupyter notebook
```

If TabPFN has not been authorized yet, the Random Forest and feature-selection sections still run normally.

In [ ]:
from __future__ import annotations

import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matbench.bench import MatbenchBenchmark
from matminer.featurizers.composition import ElementProperty
from pymatgen.core import Composition
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore', category=UserWarning)

# Make paths work whether the notebook is launched from the project root or notebooks/.
CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
RESULTS_DIR = PROJECT_ROOT / 'results'
METRICS_DIR = RESULTS_DIR / 'metrics'
FIGURES_DIR = RESULTS_DIR / 'figures'
METRICS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
TASK_NAME = 'matbench_steels'
print('Project root:', PROJECT_ROOT)
print('Results dir:', RESULTS_DIR)

## Load the Matbench task

`matbench_steels` contains 312 steel compositions and experimentally measured yield strengths in MPa. The input is composition only, so this is a good first task before moving to structure-based Matbench tasks.

In [ ]:
benchmark = MatbenchBenchmark(autoload=False, subset=[TASK_NAME])
task = list(benchmark.tasks)[0]
task.load()

print('Task metadata:')
for key in ['input_type', 'task_type', 'target', 'unit', 'n_samples']:
    value = task.metadata.get(key, task.metadata.get('num_entries'))
    print(f'  {key}: {value}')

print('\nOfficial fold numbers:', task.folds_nums)
task.df.head()

## Featurize compositions with Magpie descriptors

Following the featurization idea from Lab 2, we convert each composition string into a `pymatgen.Composition`, then compute Magpie elemental statistics using `matminer`. This turns the raw chemical formulas into a tabular feature matrix that can be used by scikit-learn and TabPFN-style models.

In [ ]:
def featurize_magpie(formulas: pd.Series, show_progress: bool = False) -> pd.DataFrame:
    compositions = formulas.map(Composition)
    feature_input = pd.DataFrame({'composition': compositions}, index=formulas.index)

    featurizer = ElementProperty.from_preset('magpie')
    features = featurizer.featurize_dataframe(
        feature_input,
        col_id='composition',
        ignore_errors=False,
        inplace=False,
        pbar=show_progress,
    )
    features = features.drop(columns=['composition'])
    features = features.apply(pd.to_numeric, errors='coerce')
    features = features.replace([np.inf, -np.inf], np.nan)
    return features

X = featurize_magpie(task.df['composition'])
y = task.df['yield strength']

print('Feature matrix shape:', X.shape)
print('Target shape:', y.shape)
X.iloc[:5, :8]

## Define evaluation helpers

The important project detail is that we evaluate on the official Matbench folds. Any preprocessing that learns from data, such as imputation or feature selection, must be fit only on the training part of each fold.

In [ ]:
def build_random_forest(n_estimators: int = 500) -> Pipeline:
    return Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('model', RandomForestRegressor(
                n_estimators=n_estimators,
                random_state=RANDOM_SEED,
                n_jobs=-1,
            )),
        ]
    )


def evaluate_random_forest(
    X_all: pd.DataFrame,
    y_all: pd.Series,
    task,
    *,
    model_label: str,
    top_k_features: int | None = None,
    n_estimators: int = 500,
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, list[str]]]:
    metrics = []
    predictions = []
    selected_feature_map: dict[str, list[str]] = {}

    for fold in task.folds_nums:
        X_train_raw, y_train = task.get_train_and_val_data(fold)
        X_test_raw, y_test = task.get_test_data(fold, include_target=True)

        X_train = X_all.loc[X_train_raw.index]
        X_test = X_all.loc[X_test_raw.index]

        if top_k_features is not None:
            selector_model = build_random_forest(n_estimators=n_estimators)
            selector_model.fit(X_train, y_train)
            importances = selector_model.named_steps['model'].feature_importances_
            selected_cols = (
                pd.Series(importances, index=X_train.columns)
                .sort_values(ascending=False)
                .head(top_k_features)
                .index
                .tolist()
            )
            X_train_model = X_train[selected_cols]
            X_test_model = X_test[selected_cols]
            selected_feature_map[f'fold_{fold}'] = selected_cols
        else:
            X_train_model = X_train
            X_test_model = X_test

        model = build_random_forest(n_estimators=n_estimators)
        model.fit(X_train_model, y_train)
        y_pred = model.predict(X_test_model)

        metrics.append({
            'task': TASK_NAME,
            'feature_set': 'magpie',
            'model': model_label,
            'fold': fold,
            'train_size': len(X_train_model),
            'test_size': len(X_test_model),
            'n_features': X_train_model.shape[1],
            'mae': mean_absolute_error(y_test, y_pred),
            'r2': r2_score(y_test, y_pred),
        })

        predictions.append(pd.DataFrame({
            'model': model_label,
            'fold': fold,
            'mbid': y_test.index,
            'y_true': y_test.to_numpy(),
            'y_pred': y_pred,
            'absolute_error': np.abs(y_test.to_numpy() - y_pred),
        }))

    return pd.DataFrame(metrics), pd.concat(predictions, ignore_index=True), selected_feature_map

## Baseline: Random Forest on all Magpie features

This is the first reproducible result for the project. It is comparable to the script-based baseline already saved in `results/metrics/`.

In [ ]:
rf_metrics, rf_predictions, _ = evaluate_random_forest(
    X,
    y,
    task,
    model_label='random_forest_all_magpie',
    top_k_features=None,
)

rf_metrics

In [ ]:
rf_summary = rf_metrics[['mae', 'r2']].agg(['mean', 'std'])
rf_summary

## Visualize baseline predictions

The parity plot shows how close predicted yield strengths are to the measured values. The fold plot makes it easier to see whether performance is stable across the official Matbench splits.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(rf_predictions['y_true'], rf_predictions['y_pred'], alpha=0.75)
min_val = min(rf_predictions['y_true'].min(), rf_predictions['y_pred'].min())
max_val = max(rf_predictions['y_true'].max(), rf_predictions['y_pred'].max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1)
axes[0].set_title('Random Forest parity plot')
axes[0].set_xlabel('Measured yield strength (MPa)')
axes[0].set_ylabel('Predicted yield strength (MPa)')

axes[1].bar(rf_metrics['fold'].astype(str), rf_metrics['mae'])
axes[1].axhline(rf_metrics['mae'].mean(), color='red', linestyle='--', label='Mean MAE')
axes[1].set_title('Fold-level MAE')
axes[1].set_xlabel('Matbench fold')
axes[1].set_ylabel('MAE (MPa)')
axes[1].legend()

plt.tight_layout()
figure_path = FIGURES_DIR / 'notebook_steels_random_forest_baseline.png'
plt.savefig(figure_path, dpi=200)
plt.show()
print('Saved figure:', figure_path)

## Extension: leakage-free feature selection

Lab 2 explored feature reduction with Random Forest and PCA. Here we test a simple version that avoids data leakage:

1. For each official fold, fit a Random Forest only on that fold's training data.
2. Rank Magpie features by feature importance.
3. Keep the top 20 features.
4. Train and evaluate a new Random Forest using only those selected features.

This asks whether a smaller feature set improves generalization on the small steel dataset.

In [ ]:
rf_top20_metrics, rf_top20_predictions, selected_features = evaluate_random_forest(
    X,
    y,
    task,
    model_label='random_forest_top20_magpie',
    top_k_features=20,
)

rf_top20_metrics

In [ ]:
comparison = pd.concat([rf_metrics, rf_top20_metrics], ignore_index=True)
comparison_summary = (
    comparison
    .groupby('model')
    .agg(
        mean_mae=('mae', 'mean'),
        std_mae=('mae', 'std'),
        mean_r2=('r2', 'mean'),
        n_features=('n_features', 'mean'),
    )
    .reset_index()
)
comparison_summary

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for model_name, df_model in comparison.groupby('model'):
    ax.plot(df_model['fold'], df_model['mae'], marker='o', label=model_name)
ax.set_title('Full Magpie RF vs top-20 feature RF')
ax.set_xlabel('Matbench fold')
ax.set_ylabel('MAE (MPa)')
ax.legend()
plt.tight_layout()
figure_path = FIGURES_DIR / 'notebook_steels_feature_selection_comparison.png'
plt.savefig(figure_path, dpi=200)
plt.show()
print('Saved figure:', figure_path)

In [ ]:
# Which features were repeatedly selected across folds?
from collections import Counter

feature_counts = Counter(feature for cols in selected_features.values() for feature in cols)
selected_feature_table = (
    pd.DataFrame(feature_counts.items(), columns=['feature', 'selected_fold_count'])
    .sort_values(['selected_fold_count', 'feature'], ascending=[False, True])
    .reset_index(drop=True)
)
selected_feature_table.head(15)

## Additional model comparison: single models and an ensemble

The project prompt encourages trying new models and comparing ensembles against single predictors. Before relying on TabPFN, we add several conventional tabular baselines:

- `RidgeCV`: a simple regularized linear model.
- `ExtraTreesRegressor`: a randomized tree ensemble.
- `HistGradientBoostingRegressor`: a boosted-tree model.
- `VotingRegressor`: an average ensemble combining Random Forest, Extra Trees, and HistGradientBoosting.

All models use the same official Matbench folds and the same Magpie feature matrix.

In [ ]:
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, VotingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler


def evaluate_sklearn_estimator(
    estimator,
    X_all: pd.DataFrame,
    task,
    *,
    model_label: str,
    scale_features: bool = False,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    metrics = []
    predictions = []

    for fold in task.folds_nums:
        X_train_raw, y_train = task.get_train_and_val_data(fold)
        X_test_raw, y_test = task.get_test_data(fold, include_target=True)

        X_train = X_all.loc[X_train_raw.index]
        X_test = X_all.loc[X_test_raw.index]

        steps = [('imputer', SimpleImputer(strategy='median'))]
        if scale_features:
            steps.append(('scaler', StandardScaler()))
        steps.append(('model', clone(estimator)))

        model = Pipeline(steps=steps)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        metrics.append({
            'task': TASK_NAME,
            'feature_set': 'magpie',
            'model': model_label,
            'fold': fold,
            'train_size': len(X_train),
            'test_size': len(X_test),
            'n_features': X_train.shape[1],
            'mae': mean_absolute_error(y_test, y_pred),
            'r2': r2_score(y_test, y_pred),
        })

        predictions.append(pd.DataFrame({
            'model': model_label,
            'fold': fold,
            'mbid': y_test.index,
            'y_true': y_test.to_numpy(),
            'y_pred': y_pred,
            'absolute_error': np.abs(y_test.to_numpy() - y_pred),
        }))

    return pd.DataFrame(metrics), pd.concat(predictions, ignore_index=True)

In [ ]:
rf_estimator = RandomForestRegressor(
    n_estimators=500,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
extra_trees_estimator = ExtraTreesRegressor(
    n_estimators=500,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
hgb_estimator = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.04,
    l2_regularization=0.01,
    random_state=RANDOM_SEED,
)
ensemble_estimator = VotingRegressor([
    ('rf', rf_estimator),
    ('extra_trees', extra_trees_estimator),
    ('hgb', hgb_estimator),
])

model_specs = [
    ('ridge_cv_all_magpie', RidgeCV(alphas=np.logspace(-3, 3, 13)), True),
    ('extra_trees_all_magpie', extra_trees_estimator, False),
    ('hist_gradient_boosting_all_magpie', hgb_estimator, False),
    ('rf_extra_trees_hgb_voting_ensemble', ensemble_estimator, False),
]

additional_metric_tables = []
additional_prediction_tables = []

for model_label, estimator, scale_features in model_specs:
    metrics_i, predictions_i = evaluate_sklearn_estimator(
        estimator,
        X,
        task,
        model_label=model_label,
        scale_features=scale_features,
    )
    additional_metric_tables.append(metrics_i)
    additional_prediction_tables.append(predictions_i)

additional_model_metrics = pd.concat(additional_metric_tables, ignore_index=True)
additional_model_predictions = pd.concat(additional_prediction_tables, ignore_index=True)
additional_model_metrics

In [ ]:
all_non_tabpfn_metrics = pd.concat([comparison, additional_model_metrics], ignore_index=True)
all_model_summary = (
    all_non_tabpfn_metrics
    .groupby('model')
    .agg(
        mean_mae=('mae', 'mean'),
        std_mae=('mae', 'std'),
        mean_r2=('r2', 'mean'),
        n_features=('n_features', 'mean'),
    )
    .sort_values('mean_mae')
    .reset_index()
)
all_model_summary

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_df = all_model_summary.sort_values('mean_mae', ascending=True)
ax.barh(plot_df['model'], plot_df['mean_mae'], xerr=plot_df['std_mae'], alpha=0.85)
ax.set_title('Model comparison on matbench_steels')
ax.set_xlabel('Mean MAE across official folds (MPa)')
ax.invert_yaxis()
plt.tight_layout()
figure_path = FIGURES_DIR / 'notebook_steels_model_comparison.png'
plt.savefig(figure_path, dpi=200)
plt.show()
print('Saved figure:', figure_path)

best_model = all_model_summary.iloc[0]
print(
    f"Best non-TabPFN model so far: {best_model['model']} "
    f"with mean MAE={best_model['mean_mae']:.2f} MPa"
)

## Optional: TabPFNRegressor comparison

This cell is intentionally optional. TabPFN requires accepting the PriorLabs license and setting `TABPFN_TOKEN` before model weights can be downloaded.

If the token is not set, the notebook skips this section and still produces the Random Forest baseline and feature-selection extension required for a complete first project result.

In [ ]:
tabpfn_metrics = None

if os.environ.get('TABPFN_TOKEN'):
    from tabpfn import TabPFNRegressor

    def evaluate_tabpfn(X_all: pd.DataFrame, y_all: pd.Series, task) -> tuple[pd.DataFrame, pd.DataFrame]:
        metrics = []
        predictions = []

        for fold in task.folds_nums:
            X_train_raw, y_train = task.get_train_and_val_data(fold)
            X_test_raw, y_test = task.get_test_data(fold, include_target=True)

            X_train = X_all.loc[X_train_raw.index]
            X_test = X_all.loc[X_test_raw.index]

            model = Pipeline(steps=[
                ('imputer', SimpleImputer(strategy='median')),
                ('model', TabPFNRegressor(
                    n_estimators=8,
                    random_state=RANDOM_SEED,
                    device='auto',
                    show_progress_bar=False,
                )),
            ])
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            metrics.append({
                'task': TASK_NAME,
                'feature_set': 'magpie',
                'model': 'tabpfn_all_magpie',
                'fold': fold,
                'train_size': len(X_train),
                'test_size': len(X_test),
                'n_features': X_train.shape[1],
                'mae': mean_absolute_error(y_test, y_pred),
                'r2': r2_score(y_test, y_pred),
            })

            predictions.append(pd.DataFrame({
                'model': 'tabpfn_all_magpie',
                'fold': fold,
                'mbid': y_test.index,
                'y_true': y_test.to_numpy(),
                'y_pred': y_pred,
                'absolute_error': np.abs(y_test.to_numpy() - y_pred),
            }))

        return pd.DataFrame(metrics), pd.concat(predictions, ignore_index=True)

    tabpfn_metrics, tabpfn_predictions = evaluate_tabpfn(X, y, task)
    display(tabpfn_metrics)
else:
    print('Skipping TabPFN: TABPFN_TOKEN is not set.')
    print('After license acceptance, run: export TABPFN_TOKEN="<your-api-key>"')

## Save notebook results

The CSV output can be used directly in the final report or presentation.

In [ ]:
base_metrics_for_save = all_non_tabpfn_metrics if 'all_non_tabpfn_metrics' in globals() else comparison
tables_to_save = [base_metrics_for_save]
if tabpfn_metrics is not None:
    tables_to_save.append(tabpfn_metrics)

model_comparison = pd.concat(tables_to_save, ignore_index=True)
metrics_path = METRICS_DIR / 'notebook_steels_model_comparison.csv'
model_comparison.to_csv(metrics_path, index=False)

summary_path = METRICS_DIR / 'notebook_steels_model_summary.csv'
(
    model_comparison
    .groupby('model')
    .agg(
        mean_mae=('mae', 'mean'),
        std_mae=('mae', 'std'),
        mean_r2=('r2', 'mean'),
        n_features=('n_features', 'mean'),
    )
    .reset_index()
    .to_csv(summary_path, index=False)
)

print('Saved fold metrics:', metrics_path)
print('Saved model summary:', summary_path)
model_comparison

## Final comparison including TabPFN

After `TABPFN_TOKEN` is available, this plot adds TabPFN to the same model comparison table. If the token is not set, the plot simply reflects the non-TabPFN models.

In [ ]:
final_summary = (
    model_comparison
    .groupby('model')
    .agg(
        mean_mae=('mae', 'mean'),
        std_mae=('mae', 'std'),
        mean_r2=('r2', 'mean'),
        n_features=('n_features', 'mean'),
    )
    .sort_values('mean_mae')
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 5))
plot_df = final_summary.sort_values('mean_mae')
ax.barh(plot_df['model'], plot_df['mean_mae'], xerr=plot_df['std_mae'], alpha=0.85)
ax.set_title('Final model comparison on matbench_steels')
ax.set_xlabel('Mean MAE across official folds (MPa)')
ax.invert_yaxis()
plt.tight_layout()
figure_path = FIGURES_DIR / 'notebook_steels_final_model_comparison.png'
plt.savefig(figure_path, dpi=200)
plt.show()
print('Saved figure:', figure_path)
final_summary

## Preliminary conclusion

The Random Forest baseline establishes a reproducible benchmark on the smallest Matbench task. The feature-selection extension tests whether reducing the Magpie descriptor set helps small-data generalization.

Next steps for the project:

1. Run the optional TabPFN section after setting `TABPFN_TOKEN`.
2. Repeat the same notebook workflow on `matbench_expt_gap` or `matbench_phonons`.
3. Compare results across tasks and decide whether TabPFN or feature selection gives the most defensible improvement.